# 2b_HM_yesno_distribution_stats

Yes/no distribution statistics for the matched HM comparison set across all three control variants. Plot generation lives in `figures/answer_distribution/answer_distribution.py`; this notebook is statistical-only.

In [ ]:
from pathlib import Path
import sys
import pandas as pd
from IPython.display import display

_NOTEBOOK_DIR = Path.cwd()
if (_NOTEBOOK_DIR / "helpers.py").exists():
    sys.path.insert(0, str(_NOTEBOOK_DIR.parent))
elif (_NOTEBOOK_DIR / "notebooks" / "helpers.py").exists():
    sys.path.insert(0, str(_NOTEBOOK_DIR))

from notebooks.helpers import (
    ROOT,
    LATEX_TABLES,
    pretty_print_path,
    hh_question_means,
    hm_question_means,
    variant_summary_table,
    pairwise_variant_correlation_table,
    grouped_pattern_table,
    scatter_correlation_table,
    blind_accuracy_summary,
    qualitative_qdf,
    attach_answer_summaries,
    hh_ranked_examples,
    variant_top_bottom_table,
    hh_degradation_table,
    top_questions_tables,
    to_latex_table,
)

from notebooks.helpers import yesno_distribution_table


In [ ]:
VARIANT_LABELS = {"C": "Original", "B": "Weaker", "A": "Pronominalized"}

human_yesno_profiles = {}
yesno_inst_tables = []
for variant in ["C", "B", "A"]:
    human_profile, table = yesno_distribution_table(condition="inst_blind", variant=variant)
    human_yesno_profiles[variant] = human_profile
    yesno_inst_tables.append(table.assign(Variant=VARIANT_LABELS[variant]))

yesno_inst = pd.concat(yesno_inst_tables, ignore_index=True)
for variant in ["C", "B", "A"]:
    print(VARIANT_LABELS[variant])
    display(human_yesno_profiles[variant])
    display(yesno_inst[yesno_inst["Variant"] == VARIANT_LABELS[variant]])

In [ ]:
out = LATEX_TABLES / "hm_yesno_distribution_inst_blind.tex"
to_latex_table(
    yesno_inst,
    out,
    "Yes/no answer-distribution statistics against the human reference across control variants (instruction-aware blind condition).",
    "tab:hm_yesno_distribution_inst_blind",
    float_formatters={"JS divergence": ".3f", "TV distance": ".3f", "Chi-square": ".2f", "p": ".1e", "Yes": ".3f", "No": ".3f", "Others": ".3f"},
)
print(pretty_print_path(out))

In [ ]:
yesno_blind_tables = []
for variant in ["C", "B", "A"]:
    _, table = yesno_distribution_table(condition="blind", variant=variant)
    yesno_blind_tables.append(table.assign(Variant=VARIANT_LABELS[variant]))

yesno_blind = pd.concat(yesno_blind_tables, ignore_index=True)
display(yesno_blind)

In [ ]:
ranked_yesno = yesno_inst.sort_values(["Variant", "JS divergence", "TV distance", "Model"]).reset_index(drop=True)
for variant in ["Original", "Weaker", "Pronominalized"]:
    sub = ranked_yesno[ranked_yesno["Variant"] == variant]
    best_js = sub.iloc[0][["Variant", "Model", "Group", "JS divergence", "TV distance", "Chi-square", "p", "Significant"]]
    worst_js = sub.iloc[-1][["Variant", "Model", "Group", "JS divergence", "TV distance", "Chi-square", "p", "Significant"]]
    print(f"Closest yes/no distribution to humans ({variant}, inst_blind):")
    display(best_js.to_frame().T)
    print(f"Farthest yes/no distribution from humans ({variant}, inst_blind):")
    display(worst_js.to_frame().T)

print("Interpretation: models are ranked by closeness to the human yes/no distribution using JS divergence and TV distance. The chi-square test is also reported as a significance test for whether a model's aggregate category counts differ from the human distribution.")